<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/3feat-encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sports14 Text/Image Feature Extraction

In [2]:
import os
import numpy as np
import pandas as pd

In [2]:
# os.chdir('/home/xin/MMRec/Sports14')
# os.getcwd()

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
PATH = "/content/drive/MyDrive/Classroom/HTTT_Projects/CD HTTT TM/MMRec/data"

In [5]:
!cp -r "{PATH}/." .

## Load text data

In [6]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
file_name = 'meta-sports14.csv'

meta_file = os.path.join(file_path, file_name)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

data loaded!
shape: (18357, 10)


,itemID,asin,title,price,imUrl,related,brand,categories,salesRank,description
0,0,1881509818,Ghost Inc Glock Armorers Tool 3/32 Punch,9.99,http://ecx.images-amazon.com/images/I/21iMxsyD...,"{'also_bought': ['B000U3YWEM', 'B000U401J6', '...",Ghost,"[['Sports & Outdoors', 'Hunting & Fishing', 'H...",{'Sports &amp; Outdoors': 172909},Ghost Armorer Tool (1). The GAT is made with a...
1,1,2094869245,5 LED Bicycle Rear Tail Red Bike Torch Laser B...,8.26,http://ecx.images-amazon.com/images/I/51RtwnJw...,"{'also_bought': ['B0081O93N2', 'B00EYTCHJA', '...",NaN,"[['Sports & Outdoors', 'Cycling', 'Lights & Re...",{'Sports &amp; Outdoors': 14293},This newly-designed Laser tail light can emit ...
2,2,7245456259,Black Mountain Products Single Resistance Band...,10.49,http://ecx.images-amazon.com/images/I/411Ikpf1...,"{'also_bought': ['B00DDBS2JE', 'B00H1KNHE8', '...",Black Mountain,"[['Sports & Outdoors', 'Exercise & Fitness', '...",{'Sports &amp; Outdoors': 1010},Black Mountain Products single resistance band...


In [7]:
# sentences: title + brand + category + description | All have title + description

title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['description'].isnull()]
print(desc_na_df.shape)

na_df = df[df['description'].isnull() & df['title'].isnull()]
print(na_df.shape)

na3_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull()]
print(na3_df.shape)

na4_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull() & df['categories'].isnull()]
print(na4_df.shape)

(91, 10)
(2659, 10)
(40, 10)
(40, 10)
(0, 10)


In [8]:
df[desc_str] = df[desc_str].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")


In [9]:
sentences = []
for i, row in df.iterrows():
    sen = row['title'] + ' ' + row['brand'] + ' '
    cates = eval(row['categories'])
    if isinstance(cates, list):
        for c in cates[0]:
            sen = sen + c + ' '
    sen += row[desc_str]
    sen = sen.replace('\n', ' ')

    sentences.append(sen)

sentences[:5]

['Ghost Inc Glock Armorers Tool 3/32 Punch Ghost Sports & Outdoors Hunting & Fishing Hunting Gun Maintenance Gunsmithing Tools Ghost Armorer Tool (1). The GAT is made with a spring steel punch. The diameter is 3/32 of an inch or 2.5mm, this is the same as the OEM tool size. The difference is you will be able to press harder without bending the shaft of this punch. Just a better tool to work on your Glock with.',
 '5 LED Bicycle Rear Tail Red Bike Torch Laser Beam Lamp Light   Sports & Outdoors Cycling Lights & Reflectors Taillights This newly-designed Laser tail light can emit two parallel lines, to form a virtual lane together with the moving of bicycle on the road. LED flash light and  two lines not only enhance the waring effect strongly and greatly but also improve the safety of night riding.',
 'Black Mountain Products Single Resistance Band - Door Anchor and Starter Guide Included Black Mountain Sports & Outdoors Exercise & Fitness Accessories Exercise Bands Black Mountain Produc

In [10]:
course_list = df[i_id].tolist()
#sentences = df[desc_str].tolist()

assert course_list[-1] == len(course_list) - 1

In [11]:
!pip install sentence_transformers

In [25]:
# should `pip install sentence_transformers` first
# should use gpu to speed up
from sentence_transformers import SentenceTransformer
# all-MiniLM-L6-v2, all-MiniLM-L12-v2, all-mpnet-base-v2 gpu (31s) (36s) (3p)
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
sentence_embeddings = model.encode(sentences)
print('text encoded!')

assert sentence_embeddings.shape[0] == df.shape[0]
np.save(os.path.join(file_path, 'text_feat.npy'), sentence_embeddings)
print('done!')

text encoded!
done!


In [27]:
sentence_embeddings[:10]

array([[-0.12623428,  0.03341388, -0.01948772, ..., -0.10133382,
         0.05145447,  0.07334711],
       [ 0.00680288,  0.00055714, -0.03157382, ...,  0.03421347,
         0.02450731,  0.03113372],
       [-0.12395922,  0.05546277, -0.00272344, ..., -0.19819075,
         0.04171505,  0.0510535 ],
       ...,
       [-0.06516662,  0.04306807, -0.00357154, ...,  0.02348826,
        -0.02514206,  0.06650117],
       [ 0.05071204,  0.03823142, -0.04340539, ...,  0.00951273,
         0.05093091,  0.03292944],
       [-0.13305897,  0.0793426 , -0.01714418, ..., -0.1128435 ,
        -0.00523037,  0.03694076]], dtype=float32)

In [22]:
sentence_embeddings[:10]

array([[-0.03981331,  0.04550591, -0.08056157, ..., -0.09733431,
         0.02406665,  0.10050276],
       [ 0.04490448,  0.07312506, -0.0612434 , ...,  0.04266131,
         0.03126093,  0.01809958],
       [-0.0448086 ,  0.00965227, -0.03126742, ..., -0.13017634,
         0.08312534,  0.01685718],
       ...,
       [ 0.02831936, -0.01067614,  0.073575  , ..., -0.01280067,
        -0.02664665,  0.04623172],
       [ 0.05305276,  0.03369725, -0.06977648, ...,  0.02138268,
         0.09511104,  0.00255868],
       [-0.01005792,  0.0669528 , -0.06504411, ..., -0.15564714,
        -0.00174886,  0.02124636]], dtype=float32)

In [18]:
sentence_embeddings[:10]

array([[ 0.0130763 , -0.07844383,  0.01342942, ...,  0.03859959,
        -0.02612614, -0.02280949],
       [ 0.04801448, -0.07518486, -0.00909536, ..., -0.01204228,
         0.00749911, -0.03410789],
       [-0.01165349, -0.08039966, -0.02328087, ...,  0.00364627,
        -0.04098817, -0.01225114],
       ...,
       [-0.025987  , -0.02239929, -0.03765765, ..., -0.00202438,
         0.01253257, -0.00203957],
       [-0.00486367, -0.07356925,  0.01529015, ..., -0.02836558,
        -0.0340668 , -0.01288448],
       [ 0.00729822, -0.09941685,  0.02093659, ..., -0.00405612,
         0.0045804 , -0.02809502]], dtype=float32)

In [11]:
# !cp "{PATH}/text_feat.all-mpnet-base-v2.npy" "text_feat.npy"

In [12]:
load_txt_feat = np.load('text_feat.npy', allow_pickle=True)
print(load_txt_feat.shape)
load_txt_feat[:10]

(18357, 768)


array([[ 0.0130763 , -0.07844383,  0.01342942, ...,  0.03859959,
        -0.02612614, -0.02280949],
       [ 0.04801448, -0.07518486, -0.00909536, ..., -0.01204228,
         0.00749911, -0.03410789],
       [-0.01165349, -0.08039966, -0.02328087, ...,  0.00364627,
        -0.04098817, -0.01225114],
       ...,
       [-0.025987  , -0.02239929, -0.03765765, ..., -0.00202438,
         0.01253257, -0.00203957],
       [-0.00486367, -0.07356925,  0.01529015, ..., -0.02836558,
        -0.0340668 , -0.01288448],
       [ 0.00729822, -0.09941685,  0.02093659, ..., -0.00405612,
         0.0045804 , -0.02809502]], dtype=float32)

In [ ]:
!cp "text_feat.npy" "{PATH}/"

In [28]:
!cp "text_feat.npy" "{PATH}/text_feat.all-MiniLM-L6-v2.npy"

# Image encoder (V0)，following LATTICE, averaging over for missed items

In [ ]:
df[:5]

,itemID,asin,title,price,imUrl,related,brand,categories,salesRank,description
0,0,1881509818,Ghost Inc Glock Armorers Tool 3/32 Punch,9.99,http://ecx.images-amazon.com/images/I/21iMxsyD...,"{'also_bought': ['B000U3YWEM', 'B000U401J6', '...",Ghost,"[['Sports & Outdoors', 'Hunting & Fishing', 'H...",{'Sports &amp; Outdoors': 172909},Ghost Armorer Tool (1). The GAT is made with a...
1,1,2094869245,5 LED Bicycle Rear Tail Red Bike Torch Laser B...,8.26,http://ecx.images-amazon.com/images/I/51RtwnJw...,"{'also_bought': ['B0081O93N2', 'B00EYTCHJA', '...",,"[['Sports & Outdoors', 'Cycling', 'Lights & Re...",{'Sports &amp; Outdoors': 14293},This newly-designed Laser tail light can emit ...
2,2,7245456259,Black Mountain Products Single Resistance Band...,10.49,http://ecx.images-amazon.com/images/I/411Ikpf1...,"{'also_bought': ['B00DDBS2JE', 'B00H1KNHE8', '...",Black Mountain,"[['Sports & Outdoors', 'Exercise & Fitness', '...",{'Sports &amp; Outdoors': 1010},Black Mountain Products single resistance band...
3,3,7245456313,Black Mountain Products Resistance Band Set wi...,32.99,http://ecx.images-amazon.com/images/I/51FdHlZS...,"{'also_bought': ['1612431712', 'B00GSBMW2Y', '...",Black Mountain,"[['Sports & Outdoors', 'Exercise & Fitness', '...",{'Sports &amp; Outdoors': 15},[if gte mso 9]><xml> <o:OfficeDocumentSettings...
4,4,B000002NUS,Outers Universal 32-Piece Blow Molded Gun Clea...,21.99,http://ecx.images-amazon.com/images/I/510GjWgd...,"{'also_bought': ['B000PW64JY', 'B0010KHNEU', '...",Outers,"[['Sports & Outdoors', 'Hunting & Fishing', 'H...",{'Sports &amp; Outdoors': 26738},Outers now offers this rigid and durable hard ...


In [ ]:
import array

def readImageFeatures(path):
  f = open(path, 'rb')
  while True:
    asin = f.read(10).decode('UTF-8')
    if asin == '': break
    a = array.array('f')
    a.fromfile(f, 4096)
    yield asin, a.tolist()

In [ ]:

img_data = readImageFeatures("image_features_Sports_and_Outdoors.b")
item2id = dict(zip(df['asin'], df['itemID']))

feats = {}
avg = []
for d in img_data:
    if d[0] in item2id:
        feats[int(item2id[d[0]])] = d[1]
        avg.append(d[1])
avg = np.array(avg).mean(0).tolist()

ret = []
non_no = []
for i in range(len(item2id)):
    if i in feats:
        ret.append(feats[i])
    else:
        non_no.append(i)
        ret.append(avg)

print('# of items not in processed image features:', len(non_no))
assert len(ret) == len(item2id)
np.save('image_feat.npy', np.array(ret))
np.savetxt("missed_img_itemIDs.csv", non_no, delimiter =",", fmt ='%d')
print('done!')

# of items not in processed image features: 180
done!
